<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Data Carpentry · Python profesional</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **MultiIndex, Pivot Tables, Joins y Merges avanzados**

En este laboratorio trabajaremos una de las áreas donde más errores silenciosos aparecen en proyectos de datos: **reestructuración y combinación de tablas**.

Seguiremos el patrón:

> **construir → observar → provocar inconsistencia → diagnosticar → corregir → validar → comparar Pandas/Polars → medir → interpretar**

El objetivo no es memorizar `merge()`, sino comprender:

- granularidad;
- claves;
- cardinalidad;
- duplicados;
- pérdida de filas;
- explosión de filas;
- valores sin correspondencia;
- estructuras jerárquicas;
- reshape;
- joins temporales;
- diferencias de modelo entre Pandas y Polars.

## Objetivos

Al finalizar deberías poder:

1. Construir y leer un `MultiIndex`.
2. Seleccionar niveles correctamente.
3. Utilizar `stack`, `unstack`, `pivot` y `pivot_table`.
4. Detectar cuándo `pivot()` falla por duplicados.
5. Elegir una agregación adecuada con `pivot_table()`.
6. Entender `merge()` y sus cardinalidades.
7. Validar relaciones `1:1`, `1:m`, `m:1` y `m:m`.
8. Detectar explosiones de filas.
9. Usar `indicator=True` para auditar joins.
10. Implementar anti-joins y semi-joins.
11. Gestionar claves duplicadas antes de unir.
12. Trabajar con joins por múltiples claves.
13. Utilizar `merge_asof()` en datos temporales.
14. Entender por qué Polars no necesita MultiIndex.
15. Realizar joins equivalentes en Polars.
16. Utilizar `semi` y `anti` joins nativos en Polars.
17. Realizar `join_asof()` en Polars.
18. Comparar rendimiento y validar equivalencia.

## Cómo trabajar

**Alumnado:** ejecuta la parte guiada y completa los diecisiete ejercicios en orden. Sustituye los TODO por código y comprobaciones. Algunos ejercicios reutilizan tablas anteriores. Ejecutar las plantillas no demuestra que la práctica esté resuelta.

**Dependencias:** Python, NumPy, Pandas, Matplotlib, Polars y PyArrow. Instala con `python -m pip install numpy pandas matplotlib polars pyarrow`. Los bloques Polars conservan la protección del original si no está instalado.

**Datos:** 300.000 ventas sintéticas, con claves huérfanas deliberadas. No se utilizan datos externos. Los joins temporales exigen ordenar las claves temporales y definir política para empates y datos sin historial. Los benchmarks no establecen un ganador universal. El docente seleccionará ejercicios de aula y ampliación.


## 1. Preparación

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

try:
    import polars as pl
    POLARS_DISPONIBLE = True
    print("Polars:", pl.__version__)
except ImportError:
    POLARS_DISPONIBLE = False
    print("Polars no instalado. Ejecuta: pip install polars")

print("Pandas:", pd.__version__)

# 2. Crear tablas relacionadas

In [ ]:
n_clientes = 30_000
n_ventas = 300_000

clientes = pd.DataFrame({
    "cliente_id": np.arange(1, n_clientes + 1),
    "segmento": rng.choice(
        ["Particular", "PyME", "Enterprise"],
        n_clientes,
        p=[0.65, 0.27, 0.08]
    ),
    "region_cliente": rng.choice(
        ["Norte", "Sur", "Este", "Oeste", "Centro"],
        n_clientes
    )
})

ventas = pd.DataFrame({
    "venta_id": np.arange(1, n_ventas + 1),
    "cliente_id": rng.integers(1, n_clientes + 1501, n_ventas),
    "region": rng.choice(
        ["Norte", "Sur", "Este", "Oeste", "Centro"],
        n_ventas
    ),
    "canal": rng.choice(
        ["web", "tienda", "partner", "marketplace"],
        n_ventas
    ),
    "producto": rng.choice(
        [f"P{i:02d}" for i in range(1, 21)],
        n_ventas
    ),
    "unidades": rng.integers(1, 8, n_ventas),
    "precio": rng.uniform(10, 300, n_ventas),
    "fecha": pd.Timestamp("2025-01-01") + pd.to_timedelta(
        rng.integers(0, 365 * 24 * 60, n_ventas),
        unit="m"
    )
})

ventas["importe"] = ventas["unidades"] * ventas["precio"]

print("Clientes:", clientes.shape)
print("Ventas:", ventas.shape)

### Inconsistencia deliberada

`ventas` contiene algunos `cliente_id` que **no existen** en la tabla maestra `clientes`.

No los corregiremos todavía. Primero veremos cómo un join permite detectarlos.

# 3. Granularidad antes de unir

In [ ]:
print("¿cliente_id único en clientes?:", clientes["cliente_id"].is_unique)
print("¿venta_id único en ventas?:", ventas["venta_id"].is_unique)
print("¿cliente_id único en ventas?:", ventas["cliente_id"].is_unique)

### Interpretación

- `clientes`: una fila por cliente.
- `ventas`: una fila por operación.
- un cliente puede tener muchas ventas.

La relación esperada al añadir atributos de cliente a ventas es **muchos-a-uno (`m:1`)**.

# 4. MultiIndex en Pandas

In [ ]:
resumen = (
    ventas.groupby(
        ["region", "canal"]
    )
    .agg(
        facturacion=("importe", "sum"),
        operaciones=("venta_id", "size")
    )
)

resumen.head(10)

## 4.1 Inspeccionar el índice

In [ ]:
print(type(resumen.index))
print("Nombres:", resumen.index.names)
print("Niveles:")
for nivel in resumen.index.levels:
    print(list(nivel))

Un `MultiIndex` representa varias dimensiones dentro del índice. Es potente, pero puede aumentar la complejidad de selección, reshape y exportación.

# 5. Seleccionar en MultiIndex

In [ ]:
resumen.loc["Norte"]

In [ ]:
resumen.loc[("Norte", "web")]

## 5.1 `xs()` para cortar por nivel

In [ ]:
resumen.xs(
    "web",
    level="canal"
)

`xs()` resulta útil para seleccionar una sección transversal de un nivel jerárquico.

# 6. Volver a columnas planas

In [ ]:
resumen_plano = resumen.reset_index()
resumen_plano.head()

En pipelines de producción suele ser más sencillo mantener dimensiones como columnas explícitas, especialmente al interoperar con SQL, Parquet o Polars.

# 7. `unstack()`

In [ ]:
facturacion_unstack = (
    ventas.groupby(
        ["region", "canal"]
    )["importe"]
    .sum()
    .unstack("canal")
)

facturacion_unstack

## 7.1 Visualizar

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for canal in facturacion_unstack.columns:
    ax.plot(
        facturacion_unstack.index,
        facturacion_unstack[canal],
        marker="o",
        label=canal
    )

ax.set(
    title="Facturación por región y canal",
    xlabel="Región",
    ylabel="Facturación"
)
ax.legend()
plt.show()

# 8. `stack()`

In [ ]:
facturacion_restack = facturacion_unstack.stack()
facturacion_restack.head(10)

`stack()` y `unstack()` desplazan dimensiones entre índice y columnas.

# 9. `pivot()` — primero un caso válido

In [ ]:
tabla_simple = pd.DataFrame({
    "region": ["Norte", "Norte", "Sur", "Sur"],
    "canal": ["web", "tienda", "web", "tienda"],
    "valor": [100, 120, 90, 150]
})

tabla_simple.pivot(
    index="region",
    columns="canal",
    values="valor"
)

# 10. Provocar fallo de `pivot()` por duplicados

In [ ]:
tabla_duplicada = pd.DataFrame({
    "region": ["Norte", "Norte", "Norte", "Sur"],
    "canal": ["web", "web", "tienda", "web"],
    "valor": [100, 80, 120, 90]
})

tabla_duplicada

In [ ]:
try:
    tabla_duplicada.pivot(
        index="region",
        columns="canal",
        values="valor"
    )
except ValueError as error:
    print(type(error).__name__, "->", error)

### Diagnóstico

`pivot()` necesita una correspondencia única para cada combinación `index × columns`.

La combinación `Norte-web` aparece dos veces, por lo que Pandas no puede decidir qué valor colocar.

# 11. Corregir con `pivot_table()`

In [ ]:
tabla_pivot = tabla_duplicada.pivot_table(
    index="region",
    columns="canal",
    values="valor",
    aggfunc="sum",
    fill_value=0
)

tabla_pivot

### Interpretación

No hemos “arreglado” técnicamente el error sin más: hemos tomado una decisión analítica explícita.

Para duplicados `region-canal`, queremos **sumar** los valores.

# 12. Pivot table real

In [ ]:
pivot_real = ventas.pivot_table(
    index="region",
    columns="canal",
    values="importe",
    aggfunc="sum",
    fill_value=0
)

pivot_real

## 12.1 Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

im = ax.imshow(
    pivot_real.values,
    aspect="auto"
)

ax.set_xticks(
    range(len(pivot_real.columns)),
    pivot_real.columns
)
ax.set_yticks(
    range(len(pivot_real.index)),
    pivot_real.index
)

ax.set_title("Facturación: región × canal")

for i in range(pivot_real.shape[0]):
    for j in range(pivot_real.shape[1]):
        ax.text(
            j, i,
            f"{pivot_real.iloc[i, j] / 1e6:.1f}M",
            ha="center",
            va="center"
        )

plt.colorbar(im, ax=ax, label="Facturación")
plt.show()

# 13. Primer merge

In [ ]:
ventas_clientes = ventas.merge(
    clientes,
    on="cliente_id",
    how="left"
)

ventas_clientes.head()

## 13.1 ¿Hemos perdido filas?

In [ ]:
print("Ventas antes:", len(ventas))
print("Filas después:", len(ventas_clientes))
print(
    "Diferencia:",
    len(ventas_clientes) - len(ventas)
)

El número de filas se mantiene, pero eso no garantiza que el join sea correcto. Debemos comprobar también las correspondencias.

# 14. Auditar con `indicator=True`

In [ ]:
auditoria_join = ventas.merge(
    clientes,
    on="cliente_id",
    how="left",
    indicator=True
)

auditoria_join["_merge"].value_counts()

## 14.1 Visualizar correspondencias

In [ ]:
conteo_merge = (
    auditoria_join["_merge"]
    .value_counts()
    .reindex(
        ["both", "left_only", "right_only"],
        fill_value=0
    )
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    conteo_merge.index.astype(str),
    conteo_merge.values
)
ax.set(
    title="Auditoría del join ventas → clientes",
    xlabel="Estado",
    ylabel="Filas"
)
plt.show()

### Interpretación

Las filas `left_only` son ventas cuyo `cliente_id` no aparece en la tabla maestra.

El join ha convertido una inconsistencia referencial en algo medible.

# 15. Inspeccionar clientes huérfanos

In [ ]:
huerfanas = auditoria_join.loc[
    auditoria_join["_merge"] == "left_only",
    ["venta_id", "cliente_id", "importe"]
]

huerfanas.head()

In [ ]:
print(
    "Ventas huérfanas:",
    len(huerfanas)
)

print(
    "Clientes desconocidos:",
    huerfanas["cliente_id"].nunique()
)

# 16. Anti-join en Pandas

In [ ]:
anti_join_pd = (
    ventas.merge(
        clientes[["cliente_id"]],
        on="cliente_id",
        how="left",
        indicator=True
    )
    .loc[
        lambda df: df["_merge"] == "left_only"
    ]
    .drop(columns="_merge")
)

anti_join_pd.head()

Un anti-join devuelve las filas de la izquierda que **no encuentran correspondencia** en la derecha.

Es una herramienta de calidad de datos extremadamente útil.

# 17. Semi-join en Pandas

In [ ]:
ids_validos = clientes["cliente_id"]

semi_join_pd = ventas.loc[
    ventas["cliente_id"].isin(ids_validos)
]

print(semi_join_pd.shape)

Un semi-join conserva filas de la izquierda que sí tienen correspondencia, pero no añade columnas de la derecha.

# 18. Validar cardinalidad con `validate`

In [ ]:
merge_validado = ventas.merge(
    clientes,
    on="cliente_id",
    how="left",
    validate="m:1"
)

print("Merge m:1 validado.")

`validate="m:1"` convierte una expectativa conceptual en una comprobación ejecutable.

# 19. Provocar una violación de cardinalidad

In [ ]:
clientes_duplicados = pd.concat(
    [
        clientes,
        clientes.iloc[[0]]
    ],
    ignore_index=True
)

print(
    clientes_duplicados[
        clientes_duplicados["cliente_id"] == 1
    ]
)

In [ ]:
try:
    ventas.merge(
        clientes_duplicados,
        on="cliente_id",
        how="left",
        validate="m:1"
    )
except pd.errors.MergeError as error:
    print(type(error).__name__, "->", error)

### Diagnóstico

La tabla derecha ya no contiene una fila única por `cliente_id`.

Sin `validate`, el join podría ejecutarse y multiplicar filas silenciosamente.

# 20. Ver la explosión de filas sin `validate`

In [ ]:
ventas_cliente_1 = ventas.loc[
    ventas["cliente_id"] == 1
]

merge_explosivo = ventas_cliente_1.merge(
    clientes_duplicados,
    on="cliente_id",
    how="left"
)

print("Antes:", len(ventas_cliente_1))
print("Después:", len(merge_explosivo))

Una duplicación en la dimensión puede multiplicar cada fila relacionada. En datasets grandes, esto puede inflar métricas y memoria sin producir una excepción.

# 21. Corregir duplicados de clave

In [ ]:
duplicados_cliente = clientes_duplicados.loc[
    clientes_duplicados["cliente_id"].duplicated(
        keep=False
    )
]

duplicados_cliente

In [ ]:
clientes_corregidos = (
    clientes_duplicados
    .drop_duplicates(
        subset="cliente_id",
        keep="first"
    )
)

print(
    "Clave única:",
    clientes_corregidos["cliente_id"].is_unique
)

En un caso real no debemos aplicar `drop_duplicates()` automáticamente: primero hay que decidir **por qué existen duplicados y cuál es la regla correcta de resolución**.

# 22. Joins por varias claves

In [ ]:
tarifas = (
    ventas[["region", "canal"]]
    .drop_duplicates()
    .assign(
        coste_servicio=lambda df: rng.uniform(
            1, 8, len(df)
        )
    )
)

tarifas.head()

In [ ]:
ventas_tarifas = ventas.merge(
    tarifas,
    on=["region", "canal"],
    how="left",
    validate="m:1"
)

ventas_tarifas[[
    "region",
    "canal",
    "importe",
    "coste_servicio"
]].head()

# 23. Sufijos cuando existen columnas con el mismo nombre

In [ ]:
clientes_con_region = clientes.rename(
    columns={
        "region_cliente": "region"
    }
)

ejemplo_sufijos = ventas.head(5).merge(
    clientes_con_region,
    on="cliente_id",
    how="left",
    suffixes=("_venta", "_cliente")
)

ejemplo_sufijos[[
    "cliente_id",
    "region_venta",
    "region_cliente"
]]

Los sufijos hacen explícito el origen de variables homónimas. Renombrar correctamente es preferible a trabajar con `_x` y `_y` sin significado.

# 24. Detectar inconsistencias entre dos atributos

In [ ]:
comparacion_region = ventas.merge(
    clientes,
    on="cliente_id",
    how="inner",
    validate="m:1"
)

comparacion_region["region_coincide"] = (
    comparacion_region["region"]
    == comparacion_region["region_cliente"]
)

comparacion_region["region_coincide"].value_counts(normalize=True)

Este ejemplo ilustra que un join también puede utilizarse para **validar coherencia entre fuentes**, no solo para añadir columnas.

# 25. Joins temporales: problema

Supongamos que cada producto cambia de tarifa a lo largo del tiempo y queremos asignar a cada venta la última tarifa conocida **anterior o igual** a la fecha de venta.

In [ ]:
fechas_tarifa = pd.date_range(
    "2025-01-01",
    periods=12,
    freq="MS"
)

tarifas_tiempo = pd.DataFrame([
    {
        "producto": producto,
        "fecha_tarifa": fecha,
        "coste_base": float(rng.uniform(5, 120))
    }
    for producto in [f"P{i:02d}" for i in range(1, 21)]
    for fecha in fechas_tarifa
])

tarifas_tiempo.head()

# 26. `merge_asof()`

In [ ]:
ventas_asof = (
    ventas[[
        "venta_id",
        "producto",
        "fecha",
        "importe"
    ]]
    .sort_values("fecha")
)

tarifas_asof = (
    tarifas_tiempo
    .sort_values("fecha_tarifa")
)

resultado_asof = pd.merge_asof(
    ventas_asof,
    tarifas_asof,
    left_on="fecha",
    right_on="fecha_tarifa",
    by="producto",
    direction="backward"
)

resultado_asof.head()

### Interpretación

`merge_asof()` no busca igualdad exacta de fecha. Busca la observación temporal más adecuada según la dirección especificada.

Este patrón aparece en:

- precios históricos;
- sensores;
- mercados financieros;
- estados de dispositivos;
- configuraciones vigentes.

# 27. Validar que no usamos tarifas futuras

In [ ]:
violaciones_temporales = (
    resultado_asof["fecha_tarifa"]
    > resultado_asof["fecha"]
).sum()

print(
    "Tarifas futuras asignadas:",
    violaciones_temporales
)

# 28. Preparar Polars

In [ ]:
if POLARS_DISPONIBLE:
    ventas_pl = pl.from_pandas(ventas)
    clientes_pl = pl.from_pandas(clientes)
    tarifas_pl = pl.from_pandas(tarifas)
    tarifas_tiempo_pl = pl.from_pandas(tarifas_tiempo)

    print(ventas_pl.head())
else:
    print("Instalar Polars para continuar la parte comparativa.")

## Polars y MultiIndex

Polars no utiliza un índice de filas como Pandas.

Las dimensiones se mantienen como columnas explícitas. Por tanto, muchos flujos basados en:

- `set_index`;
- `reset_index`;
- `xs`;
- `stack/unstack`;

se replantean mediante columnas, `group_by`, `pivot`, `unpivot`, `select` y expresiones.

# 29. GroupBy multidimensional sin MultiIndex

In [ ]:
if POLARS_DISPONIBLE:
    resumen_pl = (
        ventas_pl
        .group_by(
            ["region", "canal"]
        )
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.len()
            .alias("operaciones")
        )
        .sort(
            ["region", "canal"]
        )
    )

    print(resumen_pl.head(10))

# 30. Pivot en Polars

In [ ]:
if POLARS_DISPONIBLE:
    pivot_pl = (
        resumen_pl
        .pivot(
            on="canal",
            index="region",
            values="facturacion",
            aggregate_function="sum"
        )
        .sort("region")
    )

    print(pivot_pl)

En Polars las dimensiones siguen siendo columnas; no aparece un MultiIndex como resultado del `group_by`.

# 31. Join left en Polars

In [ ]:
if POLARS_DISPONIBLE:
    ventas_clientes_pl = (
        ventas_pl
        .join(
            clientes_pl,
            on="cliente_id",
            how="left"
        )
    )

    print(ventas_clientes_pl.head())
    print("Filas:", ventas_clientes_pl.height)

# 32. Anti-join nativo en Polars

In [ ]:
if POLARS_DISPONIBLE:
    anti_pl = (
        ventas_pl
        .join(
            clientes_pl.select("cliente_id"),
            on="cliente_id",
            how="anti"
        )
    )

    print(anti_pl.head())
    print("Ventas huérfanas:", anti_pl.height)

### Ventaja expresiva

En Pandas construimos el anti-join mediante `merge(..., indicator=True)` y filtrado.

Polars dispone directamente de `how="anti"`.

# 33. Semi-join nativo en Polars

In [ ]:
if POLARS_DISPONIBLE:
    semi_pl = (
        ventas_pl
        .join(
            clientes_pl.select("cliente_id"),
            on="cliente_id",
            how="semi"
        )
    )

    print(semi_pl.head())
    print("Filas:", semi_pl.height)

# 34. Validar anti-join Pandas vs Polars

In [ ]:
if POLARS_DISPONIBLE:
    ids_pd = set(
        anti_join_pd["venta_id"].tolist()
    )

    ids_pl = set(
        anti_pl["venta_id"].to_list()
    )

    print(
        "Mismas ventas huérfanas:",
        ids_pd == ids_pl
    )

# 35. Join por varias claves en Polars

In [ ]:
if POLARS_DISPONIBLE:
    ventas_tarifas_pl = (
        ventas_pl
        .join(
            tarifas_pl,
            on=["region", "canal"],
            how="left"
        )
    )

    print(
        ventas_tarifas_pl.select(
            "region",
            "canal",
            "importe",
            "coste_servicio"
        ).head()
    )

# 36. Join temporal `join_asof()` en Polars

In [ ]:
if POLARS_DISPONIBLE:
    ventas_asof_pl = (
        ventas_pl
        .select(
            "venta_id",
            "producto",
            "fecha",
            "importe"
        )
        .sort("fecha")
    )

    tarifas_asof_pl = (
        tarifas_tiempo_pl
        .sort("fecha_tarifa")
    )

    resultado_asof_pl = (
        ventas_asof_pl
        .join_asof(
            tarifas_asof_pl,
            left_on="fecha",
            right_on="fecha_tarifa",
            by="producto",
            strategy="backward"
        )
    )

    print(resultado_asof_pl.head())

# 37. Validar join temporal

In [ ]:
if POLARS_DISPONIBLE:
    violaciones_pl = (
        resultado_asof_pl
        .filter(
            pl.col("fecha_tarifa")
            > pl.col("fecha")
        )
        .height
    )

    print(
        "Tarifas futuras asignadas:",
        violaciones_pl
    )

# 38. Join con validación de cardinalidad en Polars

In [ ]:
if POLARS_DISPONIBLE:
    try:
        _ = (
            ventas_pl
            .join(
                clientes_pl,
                on="cliente_id",
                how="left",
                validate="m:1"
            )
        )
        print("Cardinalidad m:1 validada en Polars.")
    except Exception as error:
        print(type(error).__name__, "->", error)

La validación de cardinalidad debe formar parte de los joins críticos siempre que conozcamos la relación esperada.

# 39. Benchmark de join Pandas vs Polars

In [ ]:
def medir(funcion, repeticiones=5):
    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion()
        tiempos.append(
            time.perf_counter() - inicio
        )

    return np.array(tiempos)


def join_pandas():
    return ventas.merge(
        clientes,
        on="cliente_id",
        how="left",
        validate="m:1"
    )


def join_polars():
    if not POLARS_DISPONIBLE:
        return None

    return ventas_pl.join(
        clientes_pl,
        on="cliente_id",
        how="left",
        validate="m:1"
    )


t_pd = medir(join_pandas)

if POLARS_DISPONIBLE:
    t_pl = medir(join_polars)
else:
    t_pl = np.array([np.nan])

benchmark_join = pd.DataFrame({
    "motor": ["Pandas", "Polars"],
    "media_ms": [
        t_pd.mean() * 1000,
        t_pl.mean() * 1000
    ],
    "std_ms": [
        t_pd.std() * 1000,
        t_pl.std() * 1000
    ]
})

benchmark_join

## 39.1 Visualizar benchmark

In [ ]:
if POLARS_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(6, 4))

    ax.bar(
        benchmark_join["motor"],
        benchmark_join["media_ms"],
        yerr=benchmark_join["std_ms"],
        capsize=4
    )

    ax.set(
        title="Join m:1 — Pandas vs Polars",
        ylabel="Tiempo medio (ms)"
    )

    plt.show()

### Interpretación

No debemos convertir un benchmark aislado en una regla universal. El rendimiento depende del tamaño, tipos, cardinalidad, hardware y pipeline completo.

# 40. Benchmark anti-join

In [ ]:
def anti_pandas():
    return (
        ventas.merge(
            clientes[["cliente_id"]],
            on="cliente_id",
            how="left",
            indicator=True
        )
        .loc[
            lambda df: df["_merge"] == "left_only"
        ]
    )


def anti_polars():
    if not POLARS_DISPONIBLE:
        return None

    return ventas_pl.join(
        clientes_pl.select("cliente_id"),
        on="cliente_id",
        how="anti"
    )


t_anti_pd = medir(anti_pandas)

if POLARS_DISPONIBLE:
    t_anti_pl = medir(anti_polars)
else:
    t_anti_pl = np.array([np.nan])

benchmark_anti = pd.DataFrame({
    "motor": ["Pandas", "Polars"],
    "media_ms": [
        t_anti_pd.mean() * 1000,
        t_anti_pl.mean() * 1000
    ]
})

benchmark_anti

## 40.1 Visualizar

In [ ]:
if POLARS_DISPONIBLE:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(
        benchmark_anti["motor"],
        benchmark_anti["media_ms"]
    )
    ax.set(
        title="Anti-join — Pandas vs Polars",
        ylabel="Tiempo medio (ms)"
    )
    plt.show()

# 41. Lazy joins en Polars

In [ ]:
if POLARS_DISPONIBLE:
    consulta_lazy = (
        ventas_pl
        .lazy()
        .filter(
            pl.col("importe") > 500
        )
        .join(
            clientes_pl.lazy(),
            on="cliente_id",
            how="left"
        )
        .group_by("segmento")
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.len()
            .alias("operaciones")
        )
        .sort("facturacion")
    )

    print(
        consulta_lazy.explain(
            optimized=True
        )
    )

## 41.1 Ejecutar el plan

In [ ]:
if POLARS_DISPONIBLE:
    resultado_lazy = consulta_lazy.collect()
    print(resultado_lazy)

### Interpretación

En Polars lazy, filtro, join y agregación forman un único plan que puede optimizarse antes de ejecutarse.

# 42. Mini-práctica integrada: auditoría de clientes y ventas

Objetivo:

1. detectar ventas huérfanas;
2. enriquecer las válidas con segmento;
3. resumir facturación por segmento y canal;
4. pivotar el resultado;
5. implementar Pandas y Polars;
6. validar equivalencia.

## 42.1 Pandas

In [ ]:
ventas_validas_pd = ventas.merge(
    clientes,
    on="cliente_id",
    how="inner",
    validate="m:1"
)

resumen_final_pd = (
    ventas_validas_pd
    .groupby(
        ["segmento", "canal"],
        as_index=False
    )
    .agg(
        facturacion=("importe", "sum"),
        operaciones=("venta_id", "size"),
        clientes=("cliente_id", "nunique")
    )
    .sort_values(
        ["segmento", "canal"]
    )
    .reset_index(drop=True)
)

resumen_final_pd

## 42.2 Polars

In [ ]:
if POLARS_DISPONIBLE:
    resumen_final_pl = (
        ventas_pl
        .join(
            clientes_pl,
            on="cliente_id",
            how="inner",
            validate="m:1"
        )
        .group_by(
            ["segmento", "canal"]
        )
        .agg(
            pl.col("importe")
            .sum()
            .alias("facturacion"),

            pl.len()
            .alias("operaciones"),

            pl.col("cliente_id")
            .n_unique()
            .alias("clientes")
        )
        .sort(
            ["segmento", "canal"]
        )
    )

    print(resumen_final_pl)

## 42.3 Validación cruzada

In [ ]:
if POLARS_DISPONIBLE:
    pl_pd = (
        resumen_final_pl
        .to_pandas()
        .reset_index(drop=True)
    )

    print(
        "Facturación:",
        np.allclose(
            resumen_final_pd["facturacion"],
            pl_pd["facturacion"]
        )
    )

    print(
        "Operaciones:",
        (
            resumen_final_pd["operaciones"]
            == pl_pd["operaciones"]
        ).all()
    )

    print(
        "Clientes:",
        (
            resumen_final_pd["clientes"]
            == pl_pd["clientes"]
        ).all()
    )

## 42.4 Pivot final

In [ ]:
pivot_segmento = resumen_final_pd.pivot_table(
    index="segmento",
    columns="canal",
    values="facturacion",
    aggfunc="sum",
    fill_value=0
)

pivot_segmento

## 42.5 Visualizar

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

im = ax.imshow(
    pivot_segmento.values,
    aspect="auto"
)

ax.set_xticks(
    range(len(pivot_segmento.columns)),
    pivot_segmento.columns
)
ax.set_yticks(
    range(len(pivot_segmento.index)),
    pivot_segmento.index
)

ax.set_title(
    "Facturación por segmento y canal"
)

for i in range(pivot_segmento.shape[0]):
    for j in range(pivot_segmento.shape[1]):
        ax.text(
            j, i,
            f"{pivot_segmento.iloc[i, j] / 1e6:.1f}M",
            ha="center",
            va="center"
        )

plt.colorbar(
    im,
    ax=ax,
    label="Facturación"
)
plt.show()

### Interpretación final

Un join correcto no se valida únicamente porque “el código no da error”.

Debemos comprobar:

- granularidad;
- cardinalidad;
- filas antes/después;
- correspondencias;
- duplicados;
- claves huérfanas;
- coherencia semántica;
- equivalencia entre implementaciones.

# 43. Checklist profesional de joins

- [ ] ¿Conozco la granularidad de cada tabla?
- [ ] ¿La clave debería ser única?
- [ ] ¿He comprobado duplicados?
- [ ] ¿Sé si la relación es 1:1, 1:m, m:1 o m:m?
- [ ] ¿Uso `validate` cuando es posible?
- [ ] ¿He comparado filas antes y después?
- [ ] ¿He auditado no correspondencias?
- [ ] ¿Necesito inner, left, right o full?
- [ ] ¿En realidad necesito un semi-join?
- [ ] ¿Necesito un anti-join para calidad?
- [ ] ¿Las columnas homónimas tienen nombres claros?
- [ ] ¿He evitado una explosión m:m accidental?
- [ ] ¿El join temporal utiliza la dirección correcta?
- [ ] ¿He validado que no se asignan datos futuros?
- [ ] ¿He comparado resultados Pandas/Polars?
- [ ] ¿He medido rendimiento antes de concluir?

# 44. Ejercicios finales


## Ejercicio 1 — MultiIndex

Crea un resumen con región, canal y producto. Ordena el índice y selecciona una región con loc y un canal con xs.


In [ ]:
# TODO: resuelve el ejercicio 1: MultiIndex.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 2 — Stack/unstack

Convierte el resumen largo en tabla ancha por canal y reconstruye la Series original. Evita confundir combinaciones ausentes con ceros reales.


In [ ]:
# TODO: resuelve el ejercicio 2: Stack/unstack.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 3 — Error de pivot

Provoca el error de pivot con dos operaciones para la misma región/canal. Corrige con pivot_table usando suma, justificándola por la granularidad.


In [ ]:
# TODO: resuelve el ejercicio 3: Error de pivot.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 4 — Cardinalidad

Construye ejemplos 1:1, 1:m, m:1 y m:m. Usa validate en cada uno y explica cuántas filas produce la multiplicidad de claves.


In [ ]:
# TODO: resuelve el ejercicio 4: Cardinalidad.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 5 — validate

Provoca MergeError al pedir m:1 sobre una dimensión duplicada. Diagnostica todas las filas de la clave duplicada y corrige solo si la duplicación es idéntica.


In [ ]:
# TODO: resuelve el ejercicio 5: validate.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 6 — Explosión de filas

Muestra la multiplicación m:m y la inflación de un importe al sumar después del join. Calcula la multiplicidad esperada por clave.


In [ ]:
# TODO: resuelve el ejercicio 6: Explosión de filas.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 7 — Indicator

Audita un left join m:1 con indicator. Comprueba filas y cuenta operaciones sin correspondencia; representa los estados.


In [ ]:
# TODO: resuelve el ejercicio 7: Indicator.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 8 — Anti-join

Encuentra ventas con cliente sin correspondencia en Pandas. Conserva solo las columnas originales y comprueba contra indicator.


In [ ]:
# TODO: resuelve el ejercicio 8: Anti-join.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 9 — Semi-join

Conserva ventas con correspondencia sin añadir columnas ni multiplicar filas. Comprueba que semi y anti particionan los hechos.


In [ ]:
# TODO: resuelve el ejercicio 9: Semi-join.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 10 — Polars

Implementa anti y semi en Polars y valida los identificadores frente a Pandas.


In [ ]:
# TODO: resuelve el ejercicio 10: Polars.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 11 — Varias claves

Une un hecho por producto y canal con una dimensión cuya combinación de claves sea única. Demuestra que validar solo producto sería incorrecto.


In [ ]:
# TODO: resuelve el ejercicio 11: Varias claves.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 12 — Asof

Asigna a cada observación el último estado anterior o igual. Incluye una observación previa a cualquier estado y conserva su falta de correspondencia.


In [ ]:
# TODO: resuelve el ejercicio 12: Asof.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 13 — Validación temporal

Valida ausencia de estados futuros y que el estado asignado sea el más reciente disponible. Compara el ejemplo temporal con Polars.


In [ ]:
# TODO: resuelve el ejercicio 13: Validación temporal.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 14 — Pivot Polars

Construye una tabla ancha de facturación por región y canal equivalente a pivot_table. Ordena filas y columnas antes de comparar.


In [ ]:
# TODO: resuelve el ejercicio 14: Pivot Polars.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 15 — Benchmark

Compara left join y anti-join entre motores con cinco repeticiones, entradas iguales y conversiones fuera de la medición. Valida filas, columnas, valores y nulos antes de medir.


In [ ]:
# TODO: resuelve el ejercicio 15: Benchmark.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 16 — Lazy join

Construye filtro web → join m:1 → group_by por segmento → suma de importe. Compara collect con un pipeline equivalente de Pandas y muestra el plan.


In [ ]:
# TODO: resuelve el ejercicio 16: Lazy join.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


## Ejercicio 17 — Reto final

Crea tres tablas con al menos 300.000 hechos: ventas, clientes e historial de tarifas. Introduce y diagnostica huérfanos y una duplicación exacta. Define una política de corrección; valida m:1 y vigencia temporal. Implementa ambos motores, semi/anti, pivot y benchmark.


In [ ]:
# TODO: resuelve el ejercicio 17: Reto final.
# Añade comprobaciones de filas, claves y valores según el enunciado.
# En Polars, utiliza if POLARS_DISPONIBLE para proteger sus expresiones.


### Tu respuesta / interpretación

_Escribe aquí el diagnóstico, política y comprobaciones._


# 45. Preguntas de reflexión


1. ¿Qué ventaja y qué coste conceptual tiene MultiIndex?
2. ¿Por qué `pivot()` puede fallar?
3. ¿Qué decisión introduce `pivot_table()`?
4. ¿Qué significa cardinalidad de un join?
5. ¿Por qué un join m:m puede ser peligroso?
6. ¿Qué aporta `validate`?
7. ¿Qué detecta `indicator=True`?
8. ¿Qué diferencia hay entre semi y anti join?
9. ¿Por qué no basta con comprobar el número final de filas?
10. ¿Cuándo usar un asof join?
11. ¿Qué riesgo existe si un asof join utiliza datos futuros?
12. ¿Cómo cambia el enfoque al no existir MultiIndex en Polars?
13. ¿Por qué los anti/semi joins son especialmente útiles en calidad de datos?
14. ¿Qué ventaja aporta un join dentro de un plan lazy?
15. ¿Qué condiciones exigirías antes de migrar un join de Pandas a Polars?

## Tus respuestas

1. …
2. …
3. …
4. …
5. …
6. …
7. …
8. …
9. …
10. …
11. …
12. …
13. …
14. …
15. …
